# 🔎 PCB Quick Ground Truth Inspector

This lightweight notebook allows you to **input any PCB image path** and instantly render the ground truth annotations with:
- 🎨 **Distinct color-coded bounding boxes** (Cyan = Capacitor, Gold = Connector, Green = IC, Magenta = Elec. Cap).
- 🔄 **Automatic Roboflow Taxonomy Aliasing** (Class 1 `Capacitor Jumper` $\rightarrow$ Class 2 `Capacitor`).
- 📊 **Instant Component Counts & Breakdown** (Target components vs non-target components).
- 🔍 **Side-by-Side Comparison**: View **Target 4 Classes** vs **All 23 Raw Annotations**.

In [ ]:
# 1. Setup & Imports
import os
import glob
from pathlib import Path
from collections import Counter
import cv2
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.dpi'] = 150
print("✅ Setup completed successfully!")

In [ ]:
# 2. Class Definitions & High-Contrast Colors
RAW_CLASSES = [
    "Button", "Capacitor Jumper", "Capacitor", "Clock", "Connector",
    "Diode", "EM", "Electrolytic Capacitor", "Ferrite Bead", "IC",
    "Inductor", "Jumper", "Led", "Pads", "Pins", "Resistor",
    "Resistor Jumper", "Resistor Network", "Switch", "Test Point",
    "Transistor", "Unknown", "Variable Resistor"
]

# Evaluated Target Classes
EVAL_CLASSES = {
    0: "Capacitor", 
    1: "Connector", 
    2: "Electrolytic Capacitor", 
    3: "IC"
}

# High-Contrast Palette for Clear Visual Differentiation
CLASS_COLORS_RGB = {
    "Capacitor": (0, 220, 255),               # Cyan
    "Capacitor Jumper": (0, 220, 255),        # Cyan (mapped to Capacitor)
    "Connector": (255, 185, 0),               # Gold / Yellow-Orange
    "Electrolytic Capacitor": (255, 50, 180), # Magenta / Hot Pink
    "IC": (0, 255, 100),                      # Bright Neon Green
    "Resistor": (255, 120, 50),               # Orange
    "Resistor Network": (255, 120, 50),       # Orange
    "Resistor Jumper": (255, 120, 50),        # Orange
    "Pins": (180, 80, 255),                   # Violet / Purple
    "Pads": (120, 200, 255),                  # Light Blue
    "Transistor": (255, 230, 80),             # Yellow
    "Diode": (255, 70, 70),                   # Coral Red
    "Default": (180, 180, 180)                # Light Gray
}

def get_color(cname):
    return CLASS_COLORS_RGB.get(cname, CLASS_COLORS_RGB["Default"])

In [ ]:
# 3. Annotation Loader & Box Renderer
def find_label_file(img_path):
    p = Path(img_path)
    # 1. Look for matching labels/ folder
    candidate1 = Path(str(p).replace("/images/", "/labels/").replace("images/", "labels/")).with_suffix(".txt")
    if candidate1.exists():
        return candidate1
    # 2. Look in the same directory
    candidate2 = p.with_suffix(".txt")
    if candidate2.exists():
        return candidate2
    return None

def load_ground_truth(label_path, img_w, img_h, map_capacitor_jumpers=True):
    boxes = []
    if not label_path or not os.path.exists(label_path):
        return boxes
    raw_lines = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                raw_lines.append((int(parts[0]), [float(p) for p in parts[1:5]]))

    if not raw_lines:
        return boxes

    # Auto-detect whether file is Legacy 23-class or Unified 4-class
    is_legacy = any(cid >= 4 for cid, _ in raw_lines)

    for raw_cid, (cx, cy, bw, bh) in raw_lines:
        x1 = int(round((cx - bw / 2) * img_w))
        y1 = int(round((cy - bh / 2) * img_h))
        x2 = int(round((cx + bw / 2) * img_w))
        y2 = int(round((cy + bh / 2) * img_h))

        if is_legacy:
            if raw_cid in (1, 2):
                norm_cid = 0
                cname = "Capacitor"
                is_target = True
            elif raw_cid == 4:
                norm_cid = 1
                cname = "Connector"
                is_target = True
            elif raw_cid == 7:
                norm_cid = 2
                cname = "Electrolytic Capacitor"
                is_target = True
            elif raw_cid in (9, 22):
                norm_cid = 3
                cname = "IC"
                is_target = True
            else:
                norm_cid = raw_cid
                cname = RAW_CLASSES[raw_cid] if raw_cid < len(RAW_CLASSES) else f"Class_{raw_cid}"
                is_target = False
        else:
            if raw_cid in EVAL_CLASSES:
                norm_cid = raw_cid
                cname = EVAL_CLASSES[raw_cid]
                is_target = True
            else:
                norm_cid = raw_cid
                cname = f"Class_{raw_cid}"
                is_target = False

        boxes.append({
            "raw_id": raw_cid,
            "cls_id": norm_cid,
            "cls_name": cname,
            "bbox": (x1, y1, x2, y2),
            "is_target": is_target
        })
    return boxes

def render_boxes(img_rgb, boxes, target_only=True, show_labels=True, box_thickness=2):
    canvas = img_rgb.copy()
    h, w = canvas.shape[:2]
    for b in boxes:
        if target_only and not b["is_target"]:
            continue
        cname = b["cls_name"]
        color = get_color(cname)
        x1, y1, x2, y2 = b["bbox"]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w - 1, x2), min(h - 1, y2)
        
        # Draw rectangle
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, box_thickness)
        
        if show_labels:
            label = f"GT:{cname[:4]}" if target_only else cname
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.38, 1)
            y_label = max(y1 - 4, th + 4)
            cv2.rectangle(canvas, (x1, y_label - th - 3), (x1 + tw + 3, y_label + 3), color, -1)
            cv2.putText(canvas, label, (x1 + 1, y_label), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 0, 0), 1, cv2.LINE_AA)
    return canvas
print("✅ Helper functions loaded!")


---

In [ ]:
# 4. 🎯 Quick Inspection: Enter Any Image Path Below!
# You can change this path to any image in data_samples or elsewhere:
IMAGE_PATH = "data_samples/images/pcb45rec1_jpg.rf.f1eaf83cd0a1b6d13c34035160e1c92f.jpg"
# IMAGE_PATH = "data_samples/images/ATTIOT_Bottom_jpg.rf.8a97ad6664656973c60d95057d9d473c.jpg"

TARGET_ONLY = True              # Set True to show our 4 target classes; False to show all 23 classes
MAP_CAPACITOR_JUMPERS = True    # Set True to map Class 1 (Capacitor Jumper) -> Class 2 (Capacitor)
SHOW_LABELS = True              # Show component text tags above boxes
FIGURE_SIZE = (12, 12)          # Plot display size

# Execution
assert os.path.exists(IMAGE_PATH), f"❌ File not found: {IMAGE_PATH}"
raw_bgr = cv2.imread(IMAGE_PATH)
rgb_img = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
h, w = rgb_img.shape[:2]

lbl_file = find_label_file(IMAGE_PATH)
gt_boxes = load_ground_truth(lbl_file, w, h, map_capacitor_jumpers=MAP_CAPACITOR_JUMPERS)
vis_img = render_boxes(rgb_img, gt_boxes, target_only=TARGET_ONLY, show_labels=SHOW_LABELS)

# Statistics
target_counts = Counter(b["cls_name"] for b in gt_boxes if b["is_target"])
all_counts = Counter(b["cls_name"] for b in gt_boxes)
non_target_counts = Counter(b["cls_name"] for b in gt_boxes if not b["is_target"])

print("=" * 75)
print(f"🖼️ Image: {Path(IMAGE_PATH).name} ({w}x{h} px)")
print(f"📄 Label file: {lbl_file}")
print(f"📦 Total annotations: {len(gt_boxes)} boxes")
print(f"🎯 Target Evaluated Objects ({sum(target_counts.values())} boxes):")
for cname, cnt in sorted(target_counts.items(), key=lambda x: -x[1]):
    print(f"     • {cname:25s}: {cnt:4d} boxes")
if not TARGET_ONLY or non_target_counts:
    print(f"⚪ Non-Target Objects ({sum(non_target_counts.values())} boxes):")
    for cname, cnt in sorted(non_target_counts.items(), key=lambda x: -x[1]):
        print(f"     • {cname:25s}: {cnt:4d} boxes")
print("=" * 75)

# Display
plt.figure(figsize=FIGURE_SIZE)
plt.imshow(vis_img)
mode_title = "Target 4 Classes" if TARGET_ONLY else "All Raw Classes"
plt.title(f"Ground Truth [{mode_title}]: {Path(IMAGE_PATH).name[:45]}\n({sum(target_counts.values())} target objects | {len(gt_boxes)} total in file)", 
          fontsize=12, fontweight='bold', pad=10, color='#1E3A8A')
plt.axis("off")
plt.tight_layout()
plt.show()

---

In [ ]:
# 5. 🔍 Side-by-Side View: Target 4 Classes vs. All 23 Raw Classes
# Use this to see why some components are drawn and where all resistors/pads are!

vis_target = render_boxes(rgb_img, gt_boxes, target_only=True, show_labels=True)
vis_all = render_boxes(rgb_img, gt_boxes, target_only=False, show_labels=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Left: Target 4 Classes
axes[0].imshow(vis_target)
axes[0].set_title(f"[A] Target Evaluated Classes ({sum(target_counts.values())} objects)\nCapacitors (Cyan), Connectors (Gold), ICs (Green)", 
                  fontsize=12, fontweight='bold', color='#065F46', pad=8)
axes[0].axis("off")

# Right: All Raw Classes
axes[1].imshow(vis_all)
axes[1].set_title(f"[B] All Raw Annotations ({len(gt_boxes)} total objects)\nIncluding Resistors (Orange), Pins (Purple), Pads (Blue)", 
                  fontsize=12, fontweight='bold', color='#1E3A8A', pad=8)
axes[1].axis("off")

plt.tight_layout()
plt.show()

---

In [ ]:
# 6. 📋 Browse All 44 Available Test Boards
# Select any sample board by index (0 to 43):
SAMPLE_INDEX = 0

available_samples = sorted(glob.glob("data_samples/images/*.jpg"))
print(f"Found {len(available_samples)} sample test images in data_samples/images/:")
for idx, img_p in enumerate(available_samples):
    mark = "👉 " if idx == SAMPLE_INDEX else "   "
    print(f"{mark}[{idx:2d}] {Path(img_p).name}")

# To view a specific index, set SAMPLE_INDEX above and re-run this block:
chosen_img = available_samples[SAMPLE_INDEX]
print(f"\nCurrently selected sample [{SAMPLE_INDEX}]: {chosen_img}")
# You can now copy-paste this path into Cell 4 or Cell 5 to inspect it!

---
## 7. 🚀 Batch Inspection: Cluster Training Images vs. Roboflow GT

Verify that the **Rectified 4-Class** ground truth mapping correctly matches the **Roboflow site images**:
- 🔄 **Capacitor Aliasing**: Class 1 (`Capacitor Jumper`) + Class 2 (`Capacitor`) -> Class 0 (`Capacitor`)
- 🔌 **Connector**: Class 4 -> Class 1
- 🔋 **Electrolytic Capacitor**: Class 7 -> Class 2
- 💻 **IC**: Class 9 & Class 22 -> Class 3
- ⚪ **Non-Target Filtering**: Resistors, Pins, Pads, etc. cleanly removed

Works seamlessly **both on the YSU cluster** and **on your local machine** (auto-fetches samples via cluster SSH if not yet cached locally).


In [ ]:
# 7. Batch GT Inspector: Take 10 Train Images and Compare Raw vs Unified Mapping
import os
import subprocess
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import cv2

# Configuration
SPLIT = "train"                 # Choose 'train', 'valid', or 'test'
NUM_SAMPLES = 10                # Number of images to inspect and plot
AUTO_FETCH_IF_MISSING = True    # Set True to pull sample images from cluster if not cached
SAVE_FIGURES = True             # Set True to save high-res comparison PNGs to disk
OUTPUT_DIR = "results/cluster_gt_inspect"

# 1. Resolve Dataset Paths (Cluster vs Local Cache)
cluster_paths = [
    Path(f"/mnt/weka/etadevosyan/pcb-yolo/pcb-detection-models-training/datasets/pcb-filtered-yolov8/{SPLIT}"),
    Path(f"datasets/pcb-filtered-yolov8/{SPLIT}")
]
img_dir, raw_lbl_dir, uni_lbl_dir = None, None, None

for p in cluster_paths:
    if p.exists() and (p / "images").exists():
        img_dir = p / "images"
        raw_lbl_dir = p / "labels"
        uni_p = Path(str(p).replace("pcb-filtered-yolov8", "pcb-unified-4class"))
        uni_lbl_dir = uni_p / "labels" if uni_p.exists() else None
        print(f"📁 Using cluster dataset path: {img_dir}")
        break

if not img_dir or not img_dir.exists():
    local_cache = Path("data_samples/cluster_train")
    if (local_cache / "images").exists() and list((local_cache / "images").glob("*.jpg")):
        img_dir = local_cache / "images"
        raw_lbl_dir = local_cache / "labels_raw"
        uni_lbl_dir = local_cache / "labels_unified"
        print(f"📁 Using locally cached cluster samples: {img_dir}")
    elif AUTO_FETCH_IF_MISSING:
        print(f"📡 Fetching {NUM_SAMPLES} '{SPLIT}' samples from YSU cluster (cluster.ysu.am)...")
        from tools.inspect_cluster_gt import fetch_samples_from_cluster
        fetch_samples_from_cluster(split=SPLIT, num=NUM_SAMPLES, cache_dir="data_samples/cluster_train")
        img_dir = local_cache / "images"
        raw_lbl_dir = local_cache / "labels_raw"
        uni_lbl_dir = local_cache / "labels_unified"
    else:
        # Fallback to local data_samples
        img_dir = Path("data_samples/images")
        raw_lbl_dir = Path("data_samples/labels")
        uni_lbl_dir = None
        print(f"📁 Falling back to local data_samples: {img_dir}")

# 2. Select Sample Images
images = sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")))[:NUM_SAMPLES]
print(f"🔍 Inspecting {len(images)} '{SPLIT}' images...\n")

if SAVE_FIGURES:
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Helper function to parse YOLO labels
def parse_boxes(txt_p, w, h):
    boxes = []
    if not txt_p or not os.path.exists(txt_p):
        return boxes
    with open(txt_p, "r") as f:
        for line in f:
            pts = line.strip().split()
            if len(pts) >= 5:
                cid = int(pts[0])
                cx, cy, bw, bh = [float(x) for x in pts[1:5]]
                x1 = max(0, int(round((cx - bw / 2) * w)))
                y1 = max(0, int(round((cy - bh / 2) * h)))
                x2 = min(w - 1, int(round((cx + bw / 2) * w)))
                y2 = min(h - 1, int(round((cy + bh / 2) * h)))
                boxes.append((cid, x1, y1, x2, y2))
    return boxes

summary_stats = []

# 3. Process & Plot Each Image
for idx, img_path in enumerate(images, start=1):
    stem = img_path.stem
    raw_bgr = cv2.imread(str(img_path))
    if raw_bgr is None:
        continue
    rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    
    raw_txt = raw_lbl_dir / f"{stem}.txt" if raw_lbl_dir else None
    uni_txt = uni_lbl_dir / f"{stem}.txt" if uni_lbl_dir else None
    
    raw_boxes = parse_boxes(raw_txt, w, h)
    uni_boxes = parse_boxes(uni_txt, w, h)
    
    raw_counts = Counter(b[0] for b in raw_boxes)
    uni_counts = Counter(b[0] for b in uni_boxes)
    
    cap_jumper = raw_counts.get(1, 0)
    cap_std = raw_counts.get(2, 0)
    conn = raw_counts.get(4, 0)
    elec = raw_counts.get(7, 0)
    ic = raw_counts.get(9, 0) + raw_counts.get(22, 0)
    discarded = sum(v for k, v in raw_counts.items() if k not in (1, 2, 4, 7, 9, 22))
    
    uni_cap = uni_counts.get(0, cap_jumper + cap_std)
    uni_conn = uni_counts.get(1, conn)
    uni_elec = uni_counts.get(2, elec)
    uni_ic = uni_counts.get(3, ic)
    
    print("=" * 80)
    print(f"🖼️ [{idx:2d}/{len(images)}] Board: {img_path.name} ({w}x{h} px)")
    print(f"   • Total Raw Boxes: {len(raw_boxes):3d}  |  Total Unified Boxes: {len(uni_boxes) or (cap_jumper+cap_std+conn+elec+ic):3d}")
    print(f"   • Capacitors:    Jumper({cap_jumper:2d}) + Cap({cap_std:2d}) = {cap_jumper+cap_std:2d}  ==>  Unified Class 0: {uni_cap:2d} {'✅' if uni_cap == cap_jumper+cap_std else '❌'}")
    print(f"   • Connectors:    Raw Connector({conn:2d})               = {conn:2d}  ==>  Unified Class 1: {uni_conn:2d} {'✅' if uni_conn == conn else '❌'}")
    print(f"   • Elec. Caps:    Raw Elec Cap({elec:2d})                = {elec:2d}  ==>  Unified Class 2: {uni_elec:2d} {'✅' if uni_elec == elec else '❌'}")
    print(f"   • ICs:           Raw IC({ic:2d})                        = {ic:2d}  ==>  Unified Class 3: {uni_ic:2d} {'✅' if uni_ic == ic else '❌'}")
    print(f"   • Discarded:     Non-targets ({discarded:2d} boxes: Resistors, Pins, Pads)  ==>  Removed")
    
    summary_stats.append({
        "name": img_path.name[:32], "raw": len(raw_boxes), 
        "uni": len(uni_boxes) or (cap_jumper+cap_std+conn+elec+ic),
        "cap": uni_cap, "conn": uni_conn, "elec": uni_elec, "ic": uni_ic
    })
    
    # 4. Render Visual Plot
    vis_raw = render_boxes(rgb, [{
        "cls_name": RAW_CLASSES[b[0]] if b[0] < len(RAW_CLASSES) else f"Class_{b[0]}",
        "bbox": b[1:], "is_target": True
    } for b in raw_boxes], target_only=False, show_labels=True)
    
    RAW_TO_UNIFIED = {1: 0, 2: 0, 4: 1, 7: 2, 9: 3, 22: 3}
    boxes_to_draw = uni_boxes if uni_boxes else [(RAW_TO_UNIFIED[b[0]], *b[1:]) for b in raw_boxes if b[0] in RAW_TO_UNIFIED]
    vis_uni = render_boxes(rgb, [{
        "cls_name": EVAL_CLASSES.get(b[0], f"Class_{b[0]}"),
        "bbox": b[1:], "is_target": True
    } for b in boxes_to_draw], target_only=True, show_labels=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    axes[0].imshow(vis_raw)
    axes[0].set_title(f"[A] Roboflow Raw 23-Class GT ({len(raw_boxes)} total boxes)\n{img_path.name[:45]}", 
                      fontsize=11, fontweight="bold", color="#1E3A8A", pad=6)
    axes[0].axis("off")
    
    axes[1].imshow(vis_uni)
    axes[1].set_title(f"[B] Rectified Unified 4-Class GT ({len(boxes_to_draw)} target boxes)\nCap={uni_cap} (Cyan), Conn={uni_conn} (Gold), Elec={uni_elec} (Magenta), IC={uni_ic} (Green)", 
                      fontsize=11, fontweight="bold", color="#065F46", pad=6)
    axes[1].axis("off")
    
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(f"{OUTPUT_DIR}/gt_compare_{idx:02d}_{stem[:30]}.png", dpi=130, bbox_inches="tight")
    plt.show()

# 5. Print Overall Summary Table
print("\n" + "=" * 80)
print("📊 BATCH SUMMARY VERIFICATION TABLE:")
print("-" * 80)
print(f"{'Image Filename':<32} | {'Raw Total':>9} | {'Unified':>7} | {'Caps':>5} | {'Conn':>5} | {'Elec':>5} | {'IC':>4}")
print("-" * 80)
for r in summary_stats:
    print(f"{r['name']:<32} | {r['raw']:>9d} | {r['uni']:>7d} | {r['cap']:>5d} | {r['conn']:>5d} | {r['elec']:>5d} | {r['ic']:>4d}")
print("=" * 80)
